# Phase 3: the ablation -- what did fine-tuning actually buy? (Kaggle, resumable)

Scores the four arms from `docs/research/gui-grounding-hypothesis.md` on the v3 eval splits
(dev 555 / test_same_app 246 / test_held_out_app 790):

| arm | what it is |
|---|---|
| `uia_only` | text match against the UIA tree (scored on Windows, shipped in `results/uia_arm_results.json`) |
| `zero_shot` | base Qwen2-VL-2B, no adapter -- H1's control |
| `fine_tuned` | base + the adapter chosen on dev |
| `hybrid` | UIA when it resolves, the fine-tuned model when it doesn't |

**Protocol (fixed before any number is seen):**
1. **Choose on dev only.** Every adapter snapshot the training run saved (`adapters/step-00100` ... `final`)
   is scored on the same seeded 200-example dev subset; the best **click accuracy** wins. Zero-shot is
   scored too, as step 0 of the learning curve. The choice is pushed to the Hub (`eval/selection.json`)
   **before** a single test example is scored -- the commit history proves the order.
2. **Test once.** Zero-shot and the chosen adapter are scored on dev + both test splits.
3. **Report** every accuracy with a 95% bootstrap interval, and H1/H3 as paired differences.

**A session dying costs nothing:** every scored example goes to `eval/results/<arm>.jsonl` on the Hub
every few minutes, and a rerun skips what is already there (greedy decoding: a re-score is identical).

## Setup (same as the training notebook)
Secret `HF_TOKEN` attached (Add-ons -> Secrets), **GPU T4 x2**, Internet **On**, the v3 dataset attached.

## How to run -- Save Version -> "Save & Run All (Commit)"
| # | `RUN_MODE` | What it proves | Ends with |
|---|---|---|---|
| 1 | `"smoke"` (~15 min) | model + adapters load, answers parse, Hub writes, real s/example. **Scores dev only.** | `SMOKE PASSED` + a time estimate |
| 2 | `"eval"` (~4-6 h) | the real ablation | `EVAL FINISHED` + the H1/H3 verdicts |

If an `"eval"` session ends early, commit again: it resumes.

In [ ]:
# Cell 1 -- settings. Runs before ANY torch/transformers import, on purpose (see the training notebook).
import os
import time

SESSION_START = time.time()

RUN_MODE = "smoke"                 # "smoke" first, then "eval"
HF_REPO_NAME = "computer-use-grounder"   # the training run's repo: adapters come from it, results go back to it
CODE_REF = "a27a0814d6674148f589226d1b2933c4ea15a319"           # exact commit of rudranaresh0201/computer-use this eval runs
DEV_SELECT_SIZE = 200              # dev examples every adapter is scored on, to choose one
SELECT_SEED = 0

assert RUN_MODE in ("smoke", "eval"), RUN_MODE
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
print(f"RUN_MODE={RUN_MODE}  CODE_REF={CODE_REF}")

In [ ]:
# Cell 2 -- the same GPU type the model was trained on, checked without importing torch.
import subprocess

smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
print(smi.stdout or smi.stderr)
assert smi.returncode == 0, "No GPU attached. Session options -> Accelerator -> GPU T4 x2."
assert "T4" in smi.stdout, "Expected a Tesla T4 (Session options -> Accelerator -> GPU T4 x2)."

In [ ]:
# Cell 3 -- the token must work in THIS committed session, and must be able to write results.
import requests
from kaggle_secrets import UserSecretsClient

try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Could not read the HF_TOKEN secret. Add-ons -> Secrets: add HF_TOKEN, tick it for this notebook.") from e
assert HF_TOKEN and HF_TOKEN.startswith("hf_"), "HF_TOKEN secret is empty or not a Hugging Face token"

r = requests.get("https://huggingface.co/api/whoami-v2", headers={"Authorization": f"Bearer {HF_TOKEN}"}, timeout=30)
assert r.status_code == 200, f"Hugging Face rejected the token (HTTP {r.status_code}): {r.text[:200]}"
me = r.json()
assert me.get("auth", {}).get("accessToken", {}).get("role") != "read", "HF_TOKEN is READ-only; results need write."
HF_REPO = f"{me['name']}/{HF_REPO_NAME}"
# Smoke results live apart from the real ones, per code version, so they can never mix.
PREFIX = "eval" if RUN_MODE == "eval" else f"eval-smoke-{CODE_REF[:7]}"
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"token OK for {me['name']}; adapters from {HF_REPO}, results to {HF_REPO}/{PREFIX}/")

In [ ]:
# Cell 4 -- the exact code version, and its pinned dependencies.
import sys
from pathlib import Path

REPO_DIR = Path("/kaggle/working/computer-use")
if not REPO_DIR.is_dir():
    !git clone https://github.com/rudranaresh0201/computer-use.git {REPO_DIR}
!git -C {REPO_DIR} fetch --quiet origin
!git -C {REPO_DIR} checkout --quiet {CODE_REF}
!git -C {REPO_DIR} log -1 --format="evaluating with commit %H (%s)"
!{sys.executable} -m pip install --quiet -e "{REPO_DIR}[training]"
!{sys.executable} -m pip uninstall -y --quiet torchao

In [ ]:
# Cell 5 -- heavy imports at the pinned versions; the training run must be finished.
import importlib
import json

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
importlib.invalidate_caches()

import torch
import transformers

from computeruse.training.hub_checkpoint import HubSync

print(f"torch {torch.__version__} | transformers {transformers.__version__}")
assert transformers.__version__ == "5.0.0", "pinned transformers did not load -- restart and run from the top"
assert torch.cuda.is_available()

sync = HubSync(HF_REPO, token=HF_TOKEN)
print(f"Hub write check OK as {sync.preflight()}")
RUN_STATE = sync.read_state()
print("training run_state.json:", RUN_STATE)
assert RUN_STATE, f"{HF_REPO} has no run_state.json -- is HF_REPO_NAME the training repo?"
if RUN_MODE == "eval":
    assert RUN_STATE["finished"], "training has not finished (no final/ adapter yet) -- wait, or run smoke"
FINAL_STEP = RUN_STATE["total_steps"]

In [ ]:
# Cell 6 -- the eval data: v3 splits, labels, app richness, and the UIA arm exported on Windows.
from computeruse.dataset.registry import load_registry
from computeruse.eval.ablation import EVAL_SPLITS, load_uia_results
from computeruse.training.dataset import resolve_path
from computeruse.training.prepare_dataset import TrainingExample

candidates = sorted(Path("/kaggle/input").rglob("labels.jsonl"))
assert len(candidates) == 1, f"need exactly one labels.jsonl under /kaggle/input, found {candidates}"
DATASET_ROOT = candidates[0].parent


def read_jsonl(rel):
    return [json.loads(ln) for ln in resolve_path(DATASET_ROOT, rel).read_text(encoding="utf-8").splitlines() if ln.strip()]


SPLIT_EXAMPLES = {s: [TrainingExample(**row) for row in read_jsonl(f"splits/{s}.jsonl")] for s in EVAL_SPLITS}
sizes = {s: len(v) for s, v in SPLIT_EXAMPLES.items()}
print("eval splits:", sizes)
assert sizes == {"dev": 555, "test_same_app": 246, "test_held_out_app": 790}, "not the v3 dataset"
LABELS = {row["id"]: row for row in read_jsonl("labels.jsonl")}
RICHNESS = {c.name: c.richness for c in load_registry(REPO_DIR / "data" / "gui_grounding" / "apps.yaml")}
UIA = load_uia_results(REPO_DIR / "results" / "uia_arm_results.json")
ALL_IDS = {e.id for v in SPLIT_EXAMPLES.values() for e in v}
assert {u.example_id for u in UIA} == ALL_IDS, "UIA export and dataset disagree on the examples"
print(f"UIA arm: {len(UIA)} examples, resolved {sum(u.available for u in UIA)}")

In [ ]:
# Cell 7 -- every adapter the training run saved, on one copy of the base model.
from transformers import AutoModelForImageTextToText, AutoProcessor

from computeruse.eval.ablation import adapter_arms, adapter_path
from computeruse.eval.resumable import ResultStore
from computeruse.eval.vlm_grounder import MODEL_ID, ZERO_SHOT, arm_active, load_adapters, score_example
from computeruse.training.train_lora import resolve_precision, torch_dtype_for

ARMS = adapter_arms(sync._list_files())
print(f"adapters on the Hub: {ARMS}")
assert ARMS, "no adapters found"
if RUN_MODE == "smoke":
    ARMS = ARMS[-2:]
ADAPTER_ROOT = Path("/kaggle/working/adapters")
sync.fetch(["adapters/*/adapter_*", "final/adapter_*"], ADAPTER_ROOT)

precision = resolve_precision()
processor = AutoProcessor.from_pretrained(MODEL_ID)
base = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch_dtype_for(precision), attn_implementation="sdpa"
).to("cuda")
model = load_adapters(base, {arm: adapter_path(arm, ADAPTER_ROOT) for arm in ARMS})
del base  # from here on, the plain base model is only arm_active(model, ZERO_SHOT)
print(f"{MODEL_ID} in {precision} with {len(ARMS)} adapters; GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB")

store = ResultStore(sync, PREFIX, Path("/kaggle/working/eval_store"), flush_every_seconds=300)
print(f"results already on the Hub under {PREFIX}/: {store.pull()}")


def score(example):
    return score_example(model, processor, processor.tokenizer, example, LABELS[example.id], DATASET_ROOT)

In [ ]:
# Cell 8 -- stage 1: choose the adapter on dev ONLY, and record the choice before any test example is scored.
from computeruse.eval.ablation import choose_adapter, dev_selection_subset, learning_curve
from computeruse.eval.resumable import score_arm
from computeruse.eval.vlm_grounder import format_counts

subset = dev_selection_subset(SPLIT_EXAMPLES["dev"], 4 if RUN_MODE == "smoke" else DEV_SELECT_SIZE, seed=SELECT_SEED)
t_stage = time.time()
dev_results = {}
for arm in [ZERO_SHOT] + ARMS:
    with arm_active(model, arm), torch.no_grad():
        dev_results[arm] = score_arm(store, arm, subset, score)
CURVE = learning_curve(dev_results, FINAL_STEP)
CHOSEN = choose_adapter(CURVE)

print(f"\n{'arm':<12}{'step':>6}{'dev click acc':>15}{'parse rate':>12}{'median dist':>13}")
for row in CURVE:
    dist = "-" if row["median_center_distance"] is None else f"{row['median_center_distance']:.0f}"
    mark = "  <- chosen" if row["arm"] == CHOSEN else ""
    print(f"{row['arm']:<12}{row['step']:>6}{row['accuracy']:>15.1%}{row['parse_rate']:>12.0%}{dist:>13}{mark}")
print("\nzero-shot answer formats on this subset:", format_counts(dev_results[ZERO_SHOT]))
for r in dev_results[ZERO_SHOT][:5]:
    print(f"  zero-shot  {r.example_id}: {r.predicted_text!r} -> {r.predicted_point}, hit={r.hit}")
for r in dev_results[CHOSEN][:5]:
    print(f"  {CHOSEN:<10} {r.example_id}: {r.predicted_text!r} -> {r.predicted_point}, hit={r.hit}")

SELECTION = {"chosen": CHOSEN, "subset_size": len(subset), "seed": SELECT_SEED,
             "subset_ids": [e.id for e in subset], "curve": CURVE}
recorded = sync.fetch([f"{PREFIX}/selection.json"], Path("/kaggle/working/eval_store"))
if recorded:
    earlier = json.loads((Path("/kaggle/working/eval_store") / PREFIX / "selection.json").read_text())
    assert earlier["chosen"] == CHOSEN and earlier["subset_ids"] == SELECTION["subset_ids"], (
        f"this session chose {CHOSEN}, an earlier one recorded {earlier['chosen']} -- stop and investigate")
    print(f"\nchoice matches the one recorded earlier: {CHOSEN}")
else:
    store.push_json("selection.json", SELECTION)
    print(f"\nchoice recorded on the Hub before any test example: {CHOSEN}")

In [ ]:
# Cell 9 -- stage 2: zero-shot and the chosen adapter on every eval split, once. (Smoke: dev only.)
if RUN_MODE == "eval":
    FINAL_EXAMPLES = [e for s in EVAL_SPLITS for e in SPLIT_EXAMPLES[s]]
else:
    FINAL_EXAMPLES = subset + [e for e in SPLIT_EXAMPLES["dev"] if e not in subset][:2]
final_results = {}
for arm in (ZERO_SHOT, CHOSEN):
    with arm_active(model, arm), torch.no_grad():
        final_results[arm] = score_arm(store, arm, FINAL_EXAMPLES, score)
print(f"stage 2 done: {len(FINAL_EXAMPLES)} examples x 2 arms")

In [ ]:
# Cell 10 -- the report, and what it says.
from computeruse.eval.ablation import assemble_report

REPORT = assemble_report(
    zero_shot=final_results[ZERO_SHOT], fine_tuned=final_results[CHOSEN], chosen_arm=CHOSEN,
    richness_by_app=RICHNESS, uia_rows=UIA, curve=CURVE, selection=SELECTION,
)
Path("/kaggle/working/eval_report.json").write_text(json.dumps(REPORT, indent=2))
store.push_json("report.json", REPORT)


def cell(arm, split):
    acc = REPORT["by_arm_and_split"].get(arm, {}).get(split)
    if acc is None:
        return "-"
    lo, hi = REPORT["ci95_by_arm_and_split"][arm][split]
    return f"{acc:6.1%} [{lo:.0%},{hi:.0%}]"


print(f"\nclick accuracy, 95% CI      (fine_tuned = {CHOSEN})")
print(f"{'arm':<11}" + "".join(f"{s:>22}" for s in EVAL_SPLITS))
for arm in ("uia_only", "zero_shot", "fine_tuned", "hybrid"):
    print(f"{arm:<11}" + "".join(f"{cell(arm, s):>22}" for s in EVAL_SPLITS))


def verdict(d):
    lo, hi = d["ci95"]
    if d["n"] == 0:
        return "not measured"
    if lo > 0:
        return "SUPPORTED (interval above 0)"
    if hi < 0:
        return "CONTRADICTED (interval below 0)"
    return "inconclusive (interval includes 0)"


h1, h3 = REPORT["h1_test_same_app"], REPORT["h3_test_held_out_app"]
print(f"\nH1  trained apps, new screenshots: fine-tuned {h1['a']:.1%} vs zero-shot {h1['b']:.1%} "
      f"= {h1['diff']:+.1%} [{h1['ci95'][0]:+.1%}, {h1['ci95'][1]:+.1%}], n={h1['n']} -> {verdict(h1)}")
print(f"H3  apps never trained on:         fine-tuned {h3['a']:.1%} vs zero-shot {h3['b']:.1%} "
      f"= {h3['diff']:+.1%} [{h3['ci95'][0]:+.1%}, {h3['ci95'][1]:+.1%}], n={h3['n']} -> {verdict(h3)}")
same_vs = REPORT["h3_fine_tuned_same_vs_held_out"]
print(f"H3' fine-tuned, held-out apps vs trained apps: {same_vs['test_held_out_app']:.1%} vs {same_vs['test_same_app']:.1%}")
print("zero-shot answer formats:", {s: d["formats"] for s, d in REPORT["diagnostics"][ZERO_SHOT].items()})

hours = (time.time() - SESSION_START) / 3600
print("\n" + "=" * 70)
if RUN_MODE == "smoke":
    n_gen = sum(len(v) for v in dev_results.values()) + 2 * len(FINAL_EXAMPLES)
    per = (time.time() - t_stage) / max(n_gen, 1)
    n_arms = len(adapter_arms(sync._list_files()))
    full = (1 + n_arms) * DEV_SELECT_SIZE + 2 * (len(ALL_IDS) - DEV_SELECT_SIZE)
    print(f"SMOKE PASSED in {hours:.2f} h. ~{per:.1f} s/example here (incl. Hub pushes); the full eval is "
          f"{full} generations ~ {full * per / 3600:.1f} h. Set RUN_MODE='eval' and commit.")
else:
    print(f"EVAL FINISHED in {hours:.1f} h. Report: {HF_REPO}/{PREFIX}/report.json and /kaggle/working/eval_report.json")
print("=" * 70)